# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Shile/flyrank_ml_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import pandas as pd


In [2]:
url = 'https://raw.githubusercontent.com/Maryam-Shile/flyrank_ml_internship/refs/heads/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [11]:
df.groupby('competition_level')[['search_volume', 'competition']].agg(
    ['median', 'mean']
)

search_volume             competition          
                         median        mean      median      mean
competition_level                                                
HIGH                       20.0  291.629044        0.98  0.915854
LOW                        10.0  131.930468        0.00  0.031809
MEDIUM                     20.0  315.098039        0.48  0.481111

In [37]:
print(df[['engagement_rate', 'scroll_rate']].describe())
print(df[['scroll_rate', 'scroll_events_90d']].corr())

       engagement_rate   scroll_rate
count     30000.000000  29875.000000
mean          2.534520     18.212921
std           8.310096     29.472768
min           0.000000      0.000000
25%           0.000000      0.000000
50%           0.000000      5.000000
75%           1.350000     23.530000
max         100.000000    300.000000
                   scroll_rate  scroll_events_90d
scroll_rate           1.000000           0.027079
scroll_events_90d     0.027079           1.000000


In [18]:
impression_columns = []
for i in df.columns:
  if 'impression' in i:
    if 'tier' not in i:
      impression_columns.append(i)
  elif 'click' in i:
    impression_columns.append(i)




impression_columns

['impressions_90d',
 'clicks_90d',
 'days_with_impressions',
 'impressions_last_30d',
 'clicks_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d']

In [16]:
df.groupby('impression_tier')[impression_columns].median()

,impressions_90d,days_with_impressions,impressions_last_30d,impressions_prev_30d
impression_tier,,,,
excellent,48675.0,88.0,14329.0,17327.0
good,7249.0,88.0,1702.0,2349.0
low,31.0,17.0,3.0,7.0
moderate,998.0,86.0,198.0,296.0


In [22]:
df['ctr_tier'] = pd.qcut(df['ctr'], q = 3, duplicates = 'drop')

df.groupby('ctr_tier')[impression_columns].median()

/tmp/ipykernel_6975/3743928593.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('ctr_tier')[impression_columns].median()


,impressions_90d,clicks_90d,days_with_impressions,impressions_last_30d,clicks_last_30d,impressions_prev_30d,clicks_prev_30d
ctr_tier,,,,,,,
"(-0.001, 0.2]",399.0,0.0,71.0,65.0,0.0,110.0,0.0
"(0.2, 100.0]",2189.0,9.0,88.0,477.0,2.0,683.0,3.0


In [28]:
df[['ctr', 'pageviews_90d', 'sessions_90d', 'engaged_sessions_90d', 'scroll_events_90d']].corr()

,ctr,pageviews_90d,sessions_90d,engaged_sessions_90d,scroll_events_90d
ctr,1.000000,-0.004177,-0.006055,0.009096,0.001693
pageviews_90d,-0.004177,1.000000,0.973897,0.665992,0.547114
sessions_90d,-0.006055,0.973897,1.000000,0.679158,0.617926
engaged_sessions_90d,0.009096,0.665992,0.679158,1.000000,0.436793
scroll_events_90d,0.001693,0.547114,0.617926,0.436793,1.000000


In [33]:
tiered = ['age_tier', 'freshness_tier', 'impression_tier', 'position_tier']

for i in tiered:
  print(i)
  print(df[i].nunique())

age_tier
4
freshness_tier
4
impression_tier
4
position_tier
5


In [39]:
df_new = df[['ctr_tier', 'search_volume', 'char_count', 'provider_used', 'model_used', 'competition_level',
'age_tier', 'freshness_tier', 'position_tier', 'trend_direction', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']]
df_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   ctr_tier           30000 non-null  category
 1   search_volume      27532 non-null  float64 
 2   char_count         22301 non-null  float64 
 3   provider_used      8562 non-null   object  
 4   model_used         24267 non-null  object  
 5   competition_level  27390 non-null  object  
 6   age_tier           30000 non-null  object  
 7   freshness_tier     30000 non-null  object  
 8   position_tier      30000 non-null  object  
 9   trend_direction    30000 non-null  object  
 10  engagement_rate    30000 non-null  float64 
 11  scroll_rate        29875 non-null  float64 
 12  ai_traffic_pct     30000 non-null  float64 
dtypes: category(1), float64(5), object(7)
memory usage: 2.8+ MB


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
df['main_intent_x_content_type'] = df['main_intent'] + '_' + df['content_type']
df['main_intent_x_content_type'].value_counts()


,count
main_intent_x_content_type,
informational_keyword article,16538
transactional_keyword article,5733
commercial_keyword article,4612
informational_comparison article,697
navigational_keyword article,46


In [7]:
df.drop(columns = ['main_intent', 'content_type'], axis = 1, inplace = True)


info_key = df[df['main_intent_x_content_type'] == 'informational_keyword article'].copy()
trans_key = df[df['main_intent_x_content_type'] == 'transactional_keyword article'].copy()
com_key = df[df['main_intent_x_content_type'] == 'commercial_keyword article'].copy()
info_comp = df[df['main_intent_x_content_type'] == 'informational_comparison article'].copy()
nav_key = df[df['main_intent_x_content_type'] == 'navigational_keyword article'].copy()

all_group = [info_key, trans_key, com_key, info_comp, nav_key]


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.